In [1]:
from pathlib import Path
import os

# Notebook está en RAG/notebooks - subimos un nivel
ROOT = Path.cwd().parent
os.chdir(ROOT)

print("Project root:", Path.cwd())

import pandas as pd
from src.pipeline import answer_question


Project root: c:\Users\nicol\OneDrive\Documentos\Cursos\RAG


c:\Users\nicol\OneDrive\Documentos\Cursos\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1635.32it/s]


In [2]:
qa = pd.read_json("data/raw/financebench/financebench_open_source.jsonl", lines=True)

# En vez de un sample aleatorio: repetir las preguntas del pilot anterior
# cuya pagina de evidencia NO fue recuperada (evidence_page_found == False)
prev = pd.read_json("data/processed/pilot_results.jsonl", lines=True)
failed_ids = prev.loc[~prev["evidence_page_found"], "financebench_id"].unique().tolist()
print(f"Preguntas a repetir (evidence_page_found=False): {len(failed_ids)}")
print(failed_ids)

sample = qa[qa["financebench_id"].isin(failed_ids)].copy()

results = []
for _, row in sample.iterrows():
    resultado = answer_question(row["question"])
    expected_pages = [e["evidence_page_num"] for e in row["evidence"]]
    retrieved_pages = [s["page_num"] for s in resultado["sources"]]

    results.append({
        "financebench_id": row["financebench_id"],
        "question_type": row["question_type"],
        "company": row["company"],
        "question": row["question"],
        "expected_answer": row["answer"],
        "generated_answer": resultado["answer"],
        "expected_evidence_pages": expected_pages,
        "retrieved_pages": retrieved_pages,
        "evidence_page_found": any(p in retrieved_pages for p in expected_pages),
    })
    print(f"[{row['financebench_id']}] listo")


Preguntas a repetir (evidence_page_found=False): 10
['financebench_id_00005', 'financebench_id_01028', 'financebench_id_01091', 'financebench_id_00720', 'financebench_id_02987', 'financebench_id_03029', 'financebench_id_01858', 'financebench_id_02024', 'financebench_id_01928', 'financebench_id_00494']
[empresa='3M', año=2018, filtro_aplicado=True]
[financebench_id_03029] listo
[empresa='3M', año=None, filtro_aplicado=True]
[financebench_id_01858] listo
[empresa='Activision Blizzard', año=2019, filtro_aplicado=True]
[financebench_id_02987] listo
[empresa='Amcor', año=2023, filtro_aplicado=True]
[financebench_id_01928] listo
[empresa='American Express', año=2022, filtro_aplicado=True]
[financebench_id_01028] listo
[empresa='American Express', año=2022, filtro_aplicado=True]
[financebench_id_00720] listo
[empresa='Boeing', año=2022, filtro_aplicado=True]
[financebench_id_01091] listo
[fallback aplicado: se descartó el filtro de año, solo empresa='Boeing']
[empresa='Boeing', año=2023, filt

In [3]:
df_results = pd.DataFrame(results)
# Guardar en un archivo aparte para no sobrescribir el pilot original
df_results.to_json("data/processed/pilot_results_retry.jsonl", orient="records", lines=True)

print("\n=== Resumen (retry de preguntas fallidas) ===")
print(f"Preguntas evaluadas: {len(df_results)}")
print(f"Pagina de evidencia recuperada: {df_results['evidence_page_found'].sum()}/{len(df_results)}")
print("Revisa manualmente generated_answer vs expected_answer en data/processed/pilot_results_retry.jsonl")



=== Resumen (retry de preguntas fallidas) ===
Preguntas evaluadas: 10
Pagina de evidencia recuperada: 1/10
Revisa manualmente generated_answer vs expected_answer en data/processed/pilot_results_retry.jsonl
